# <a id='toc1_'></a>MLP with hyperparameter Tuning [(medium-article)](https://medium.com/@vijaikdasari/multilayer-perceptron-and-its-hyperparameter-tuning-deedfbca2d1b)  [&#8593;](#toc0_)

- Grid search --> all combinations, smaller networks + limited hyperparameter ranges
- Random search --> random combinations, high-dimensional hyperparameter spaces
- Bayesian optimization --> probabilistic model that balances exploration (trying new hyperparameter combinations) and exploitation (refining around known good values), complex model + large datasets
- Automated Machine Learning (AutoML) Tools --> flexible hyperparameter search configurations, Keras Tuner / Optuna

**Table of contents**<a id='toc0_'></a>    
- [MLP with hyperparameter Tuning (medium-article) ](#toc1_)    
  - [Import](#toc1_1_)    
  - [Setup](#toc1_2_)    
  - [Upload](#toc1_3_)    
  - [Splitting + augmentation + saving](#toc1_4_)    
  - [Scaling](#toc1_5_)    
  - [Definition of search space = objective function for RandomSearch and Optuna](#toc1_6_)    
  - [Future preds - REFITTING ON FULL HISTORY WITH SANITIZED TAIL](#toc1_7_)    
  - [Future preds - SANITIZED TAIL METHOD](#toc1_8_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## <a id='toc1_1_'></a>[Import](#toc0_)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from src.config import RANDOM_SEED, set_seeds
import src.pipeline as pipe
import src.models.nn as nn
import src.evaluation as eval
import src.visualization as visual
import src.reporting as rep
import pickle
import time
import numpy as np
import tensorflow as tf
import optuna
from optuna.samplers import TPESampler, RandomSampler
from sklearn.preprocessing import MinMaxScaler

C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\src\visualization\plots.py:270: SyntaxWarning: invalid escape sequence '\m'
  ax[1].plot(x, p, 'k', linewidth=2, label=f'Norm ($\mu$={mu:.2f})')


In [3]:
set_seeds()

Random seeds set to 42. Deterministic operations enabled.


## <a id='toc1_2_'></a>[Setup](#toc0_)

In [4]:
# INDICATORS = [i for i in df.columns if i != 'Year']
# per ora lo faccio solo su sti tre per capire se funziona
INDICATORS = ['cerealland_abs', 'food_production_index']
# 'fertilizer_percent'
# 'agriland_percent', 'arableland_percent'

DF_CATEGORIES = ["orig", "step_aug", "jitter_aug"]

## <a id='toc1_3_'></a>[Upload](#toc0_)

In [5]:
df = pipe.load_data()
display(df.head())

Dropped 9 unusable indicators.
Dataset loaded: 65 years (from 1960 to 2024), 27 variables.


,population_percent,population_growth,population_abs,employment_tot,employment_male,employment_female,forestarea_percent,forestarea_abs,agriland_percent,agriland_abs,...,fertilizer_percent,livestock_production_index,food_production_index,crop_production_index,cereal_production,cerealyield_abs,valueadded_percent,valueadded_dollars,exports_percent,imports_percent
1960-01-01,40.639,NaN,20400656.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1961-01-01,40.144,-0.557139,20287312.0,NaN,NaN,NaN,NaN,NaN,70.324028,206830.0,...,72.707716,70.91,85.93,93.68,13933400.0,2181.5,NaN,NaN,NaN,NaN
1962-01-01,39.645,-0.574190,20171158.0,NaN,NaN,NaN,NaN,NaN,70.218626,206520.0,...,70.071359,72.59,86.90,94.43,14433210.0,2225.3,NaN,NaN,2.563660,16.566057
1963-01-01,39.147,-0.534554,20063620.0,NaN,NaN,NaN,NaN,NaN,69.735813,205100.0,...,63.883735,65.95,86.37,97.19,13324660.0,2115.2,NaN,NaN,2.651714,14.571604
1964-01-01,38.650,-0.455075,19972523.0,NaN,NaN,NaN,NaN,NaN,69.572609,204620.0,...,64.593354,69.57,90.28,101.31,14007520.0,2243.0,NaN,NaN,2.792923,15.115329


## <a id='toc1_4_'></a>[Splitting + augmentation + saving](#toc0_)
- splitting della time serie originale in train, validation e test set,
- augmentation su train set originale (prevent data leakage) con step function e linear interpolation + jitter,
- creazione di un dizionario che contiene tutte queste cose

l'orizzonte di test è adattivo in base alla disponibilità dei dati storici di ciascun indicatore. Siccome serie sono corte ho dato priorità ad avere un punto in più nel validation set piuttosto che nel test set

In [16]:
# dizionario per salvare in memoria i subset originali e i train aumentati con step e linear
orig_aug_subsets = {}

for col in df.columns:
    print('='*30, col.upper(), '='*30)
    subset = df[[col]].copy()
    subset['Year'] = subset.index.year
    subset.rename(columns={col: 'Value'}, inplace=True)
    subset = subset[['Year', 'Value']]
    subset = subset.reset_index(drop=True)
    subset = subset.dropna()
    train_orig, val_orig, test_orig = pipe.split_train_val_test(subset)
    
    # faccio augmentation SOLO sul train set: aumento sia anni che valori
    x_train_vals = train_orig['Year']
    y_train_vals = train_orig['Value']
    
    df_step = pipe.augment_step_function(x_train_vals, y_train_vals, scale_factor=10)
    df_jitter = pipe.augment_linear_with_jitter(x_train_vals, y_train_vals, scale_factor=10, noise_level=0.05)
    visual.plot_augmented(col, df_step, df_jitter, x_train_vals, y_train_vals)
    orig_aug_subsets[col] = {
        'full_orig': subset,
        'orig_train': train_orig,
        'step_aug_train': df_step,
        'jitter_aug_train': df_jitter,
        'orig_val': val_orig,
        'orig_test': test_orig
    }

============================== POPULATION_PERCENT ==============================
Train: 1960-2004 (45 obs)
Val:   2005-2014 (10 obs)
Test:  2015-2024 (10 obs)
Grafico salvato in: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\AUGMENTATION\AUG_population_percent.png
============================== POPULATION_GROWTH ==============================
Train: 1961-2004 (44 obs)
Val:   2005-2015 (11 obs)
Test:  2016-2024 (9 obs)
Grafico salvato in: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\AUGMENTATION\AUG_population_growth.png
============================== POPULATION_ABS ==============================
Train: 1960-2004 (45 obs)
Val:   2005-2014 (10 obs)
Test:  2015-2024 (10 obs)
Grafico salvato in: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\AUGMENTATION\AUG_population_abs.png
============================== EMPLOYMENT_TOT ==============================
Train: 1991-2013 (23 obs)
Val:   2014-2018 (5 obs)
Test:  

## <a id='toc1_5_'></a>[Scaling + saving](#toc0_)
- scaling di tutto ciò che è all'interno del dizionario facendo fit su valori del train originale (diviso per anno e valore) e transform sui dataframe aumentati.

In [ ]:
scaled_orig_aug_subsets = {} 
scalers_subsets = {}

for indicator in INDICATORS:
    scaled_subsets, scalers = pipe.scale_datasets(orig_aug_subsets[indicator])
    scaled_orig_aug_subsets[indicator] = scaled_subsets
    scalers_subsets[indicator] = scalers

{'full_orig':     Year     Value
0   1960       NaN
1   1961  0.959067
2   1962  1.000000
3   1963  0.922662
4   1964  0.900023
..   ...       ...
60  2020 -0.441511
61  2021 -0.455345
62  2022 -0.441972
63  2023       NaN
64  2024       NaN

[65 rows x 2 columns], 'orig_train':     Year     Value
0   1960       NaN
1   1961  0.959067
2   1962  1.000000
3   1963  0.922662
4   1964  0.900023
5   1965  0.821856
6   1966  0.794761
7   1967  0.702440
8   1968  0.779509
9   1969  0.766100
10  1970  0.741901
11  1971  0.597213
12  1972  0.522819
13  1973  0.499151
14  1974  0.495910
15  1975  0.435246
16  1976  0.443725
17  1977  0.177961
18  1978  0.440844
19  1979  0.437409
20  1980  0.429384
21  1981  0.386149
22  1982  0.431195
23  1983  0.436199
24  1984  0.414310
25  1985  0.309356
26  1986  0.321133
27  1987  0.256407
28  1988  0.205290
29  1989  0.228746
30  1990  0.140082
31  1991  0.135236
32  1992  0.062090
33  1993  0.000000
34  1994  0.011606
35  1995  0.058285
36  1996  0.06135

## <a id='toc1_6_'></a>[Definition of search space = objective function for RandomSearch and Optuna](#toc0_)

In [8]:
def objective(trial, scaled_data_dict):
    """
    Defines an objective function that is used in hyperparameter optimization, specifically with Optuna.
    
    Args: 
        trial = which is an Optuna Trial object used to sample hyperparameters
        scaled_data_dict = contains scaled training and validation data
        
    Returns:
        best_val_loss = minimum validation loss gotten
    """
    
    params = {
        # iperparametri riguardanti la struttura della rete
        'n_layers': trial.suggest_int('n_layers', 1, 3),
        'units': trial.suggest_int('units', 8, 64),
        'activation': trial.suggest_categorical('activation', ['relu']),
        
        # iperparametri riguardanti traning
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [16, 32]),
        'dropout': trial.suggest_categorical('dropout', [0.1, 0.2, 0.3])
    }
    
    # iperparametri riguardanti i dati
    n_input_steps = trial.suggest_categorical('n_input_steps', [3, 5, 8])
    
    train_data = scaled_data_dict['train'] 
    val_data = scaled_data_dict['val']
    
    X_train, y_train = pipe.create_sequences(train_data['Value'], n_input_steps)
    X_val, y_val = pipe.create_sequences(val_data['Value'], n_input_steps)
    
    # Check di sicurezza: se la window è troppo grande e i dati pochi, X è vuoto
    if len(X_train) == 0 or len(X_val) == 0:
        raise optuna.TrialPruned() # Abortisci questo tentativo

    model = nn.build_mlp_model(params, input_dim=n_input_steps)
    history, model = nn.train_model(    
        model, 
        X_train, y_train, 
        X_val, y_val,
        batch_size=params['batch_size'],
        epochs=50,
        patience=10
        # extra_callbacks=[prune_cb]
    )
    
    # RETURNS SCORE = lowest validation loss gotten
    if 'val_loss' not in history.history:
        return float('inf')
        
    best_val_loss = min(history.history['val_loss'])
    return best_val_loss

In [9]:
from optuna.trial import TrialState
tuning_results = []
BACKUP_FILE = "tuning_results_backup.pkl"
N_TRIALS = 20

for SAMPLER_MODE in ["Random", "TPE"]:
    for indicator in INDICATORS:
        print(f"\n{'='*40} PROCESSING: {indicator} [{SAMPLER_MODE}] {'='*40}")
        for df_category in DF_CATEGORIES:
            print(f"\nOptimizing: {df_category}")
            if SAMPLER_MODE == "Random":
                sampler = RandomSampler(seed=RANDOM_SEED)
            else:
                sampler = TPESampler(seed=RANDOM_SEED)
        
            try:
                data_package = {
                    'train': scaled_orig_aug_subsets[indicator][f'{df_category}_train'],
                    'val': scaled_orig_aug_subsets[indicator]['orig_val']
                }
            except KeyError as e:
                print(f"Data Error: dataset {e} missing for {indicator}. Skipping.")
                continue

            study = optuna.create_study(direction='minimize', sampler=sampler)
            try:
                study.optimize(lambda trial: objective(trial, data_package), n_trials=N_TRIALS)
            except Exception as e:
                print(f"Optuna crash on {indicator} - {df_category}: {e}")
                continue
            
            complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])
            
            if len(complete_trials) == 0:
                print(f"!!! WARNING: no trial completed for {indicator} ({df_category}).")
                continue
            
            result_entry = {
                "indicator": indicator,
                "sampler": SAMPLER_MODE,
                "dataset_type": df_category,
                "best_params": study.best_params,
                "best_val_loss": study.best_value,
                "n_trials": N_TRIALS
            }
            
            tuning_results.append(result_entry)
            print(f"-> Saved params for {indicator} ({df_category})")

            with open(BACKUP_FILE, "wb") as f:
                pickle.dump(tuning_results, f)

print(f"\nOptimal parameters found for {len(tuning_results)} configurations.")

[I 2026-01-30 10:41:35,493] A new study created in memory with name: no-name-43c0add4-0e20-4d33-8a52-d52214621520



======================================== PROCESSING: cerealland_abs [Random] ========================================

Optimizing: orig


[I 2026-01-30 10:41:38,212] Trial 0 finished with value: 0.08816404640674591 and parameters: {'n_layers': 2, 'units': 62, 'activation': 'relu', 'learning_rate': 0.0029106359131330704, 'batch_size': 16, 'dropout': 0.3, 'n_input_steps': 5}. Best is trial 0 with value: 0.08816404640674591.
[I 2026-01-30 10:41:40,567] Trial 1 finished with value: 0.06808574497699738 and parameters: {'n_layers': 3, 'units': 55, 'activation': 'relu', 'learning_rate': 0.00026587543983272726, 'batch_size': 32, 'dropout': 0.2, 'n_input_steps': 5}. Best is trial 1 with value: 0.06808574497699738.
[I 2026-01-30 10:41:42,435] Trial 2 finished with value: 0.057678818702697754 and parameters: {'n_layers': 1, 'units': 28, 'activation': 'relu', 'learning_rate': 0.000816845589476017, 'batch_size': 16, 'dropout': 0.2, 'n_input_steps': 3}. Best is trial 2 with value: 0.057678818702697754.
[I 2026-01-30 10:41:45,120] Trial 3 finished with value: 0.10922180861234665 and parameters: {'n_layers': 3, 'units': 63, 'activation'

-> Saved params for cerealland_abs (orig)

Optimizing: step_aug


[I 2026-01-30 10:42:38,176] Trial 0 finished with value: 0.003621175419539213 and parameters: {'n_layers': 2, 'units': 62, 'activation': 'relu', 'learning_rate': 0.0029106359131330704, 'batch_size': 16, 'dropout': 0.3, 'n_input_steps': 5}. Best is trial 0 with value: 0.003621175419539213.
[I 2026-01-30 10:42:51,362] Trial 1 finished with value: 0.09143481403589249 and parameters: {'n_layers': 3, 'units': 55, 'activation': 'relu', 'learning_rate': 0.00026587543983272726, 'batch_size': 32, 'dropout': 0.2, 'n_input_steps': 5}. Best is trial 0 with value: 0.003621175419539213.
[I 2026-01-30 10:42:55,832] Trial 2 finished with value: 0.012892616912722588 and parameters: {'n_layers': 1, 'units': 28, 'activation': 'relu', 'learning_rate': 0.000816845589476017, 'batch_size': 16, 'dropout': 0.2, 'n_input_steps': 3}. Best is trial 0 with value: 0.003621175419539213.
[I 2026-01-30 10:43:04,094] Trial 3 finished with value: 0.019408797845244408 and parameters: {'n_layers': 3, 'units': 63, 'activat

...

-> Saved params for food_production_index (orig)

Optimizing: step_aug


[I 2026-01-30 11:07:14,799] Trial 0 finished with value: 0.0061119599267840385 and parameters: {'n_layers': 2, 'units': 62, 'activation': 'relu', 'learning_rate': 0.0029106359131330704, 'batch_size': 16, 'dropout': 0.3, 'n_input_steps': 5}. Best is trial 0 with value: 0.0061119599267840385.
[I 2026-01-30 11:07:20,224] Trial 1 finished with value: 0.014293545857071877 and parameters: {'n_layers': 3, 'units': 55, 'activation': 'relu', 'learning_rate': 0.00026587543983272726, 'batch_size': 32, 'dropout': 0.2, 'n_input_steps': 5}. Best is trial 0 with value: 0.0061119599267840385.
[I 2026-01-30 11:07:24,584] Trial 2 finished with value: 0.013127249665558338 and parameters: {'n_layers': 1, 'units': 28, 'activation': 'relu', 'learning_rate': 0.000816845589476017, 'batch_size': 16, 'dropout': 0.2, 'n_input_steps': 3}. Best is trial 0 with value: 0.0061119599267840385.
[I 2026-01-30 11:07:35,799] Trial 3 finished with value: 0.016010023653507233 and parameters: {'n_layers': 3, 'units': 63, 'ac

-> Saved params for food_production_index (step_aug)

Optimizing: jitter_aug


[I 2026-01-30 11:09:48,840] Trial 0 finished with value: 0.011144175194203854 and parameters: {'n_layers': 2, 'units': 62, 'activation': 'relu', 'learning_rate': 0.0029106359131330704, 'batch_size': 16, 'dropout': 0.3, 'n_input_steps': 5}. Best is trial 0 with value: 0.011144175194203854.
[I 2026-01-30 11:09:54,361] Trial 1 finished with value: 0.014773634262382984 and parameters: {'n_layers': 3, 'units': 55, 'activation': 'relu', 'learning_rate': 0.00026587543983272726, 'batch_size': 32, 'dropout': 0.2, 'n_input_steps': 5}. Best is trial 0 with value: 0.011144175194203854.
[I 2026-01-30 11:09:59,744] Trial 2 finished with value: 0.015509898774325848 and parameters: {'n_layers': 1, 'units': 28, 'activation': 'relu', 'learning_rate': 0.000816845589476017, 'batch_size': 16, 'dropout': 0.2, 'n_input_steps': 3}. Best is trial 0 with value: 0.011144175194203854.
[I 2026-01-30 11:10:14,144] Trial 3 finished with value: 0.01370226964354515 and parameters: {'n_layers': 3, 'units': 63, 'activat

-> Saved params for food_production_index (jitter_aug)

Optimal parameters found for 12 configurations.


In [17]:
predictions_dict = {}
print(f"Refitting on {len(tuning_results)} configurations.")

tf.keras.utils.set_random_seed(RANDOM_SEED)
tf.config.experimental.enable_op_determinism()

for entry in tuning_results:
    ind = entry['indicator']
    sampler = entry['sampler']
    ds_type = entry['dataset_type']
    params = entry['best_params']
    
    print(f"\nRefitting: {ind} | {sampler} | {ds_type} ...")
    
    scaler_y = scalers_subsets[ind]['scaler_y']
    train_df = scaled_orig_aug_subsets[ind][f'{ds_type}_train']
    val_df = scaled_orig_aug_subsets[ind]['orig_val']
    test_df = scaled_orig_aug_subsets[ind]['orig_test']
    
    n_steps = params['n_input_steps']
    X_train, y_train = pipe.create_sequences(train_df['Value'], n_steps)
    X_val, y_val = pipe.create_sequences(val_df['Value'], n_steps)
    X_test, y_test = pipe.create_sequences(test_df['Value'], n_steps)
    
    if len(X_test) == 0:
        print(f"SKIP {ind}-{ds_type}: test set is too short for window {n_steps}")
        continue
    
    start_time = time.time()
    try:
        model = nn.build_mlp_model(params, input_dim=n_steps)
        history, model = nn.train_model(
            model, X_train, y_train, X_val, y_val, 
            batch_size=params['batch_size'], epochs=150, patience=15
        )
    except Exception as e:
        print(f"Forecast failed for {col}: {e}")
        continue
    elapsed_time = time.time() - start_time
    
    y_pred_scaled = model.predict(X_test, verbose=0)
    
    y_pred_real = scaler_y.inverse_transform(y_pred_scaled).flatten()
    y_test_real = scaler_y.inverse_transform(y_test).flatten()
    
    aligned_years = test_df['Year'].values[n_steps:]
    
    assert len(aligned_years) == len(y_test_real), "ERROR: misalignment data"

    if ind not in predictions_dict: predictions_dict[ind] = {}
    if sampler not in predictions_dict[ind]: predictions_dict[ind][sampler] = {}
    
    predictions_dict[ind][sampler][ds_type] = {
        'years': aligned_years,
        'y_true': y_test_real,
        'y_pred': y_pred_real,
        'params': params,
        'metrics': eval.compute_errors(y_test_real, y_pred_real)
    }
    
    try:
        pred_residuals = eval.compute_residual_diagnostics(y_test_real, y_pred_real)
        print(f"  -> Residuals: Mean={pred_residuals.get('residual_mean', 0):.4f}, "
            f"Shapiro P={pred_residuals.get('shapiro_wilk_pvalue', 0):.4f}, "
            f"Ljung-Box P={pred_residuals.get('ljung_box_pvalue', 0):.4f}")
    except Exception as e:
        print(f"  -> Residual diagnostics failed: {e}")
        pred_residuals = {}
        
    try:
        visual.plot_residuals(
            y_true=y_test_real,
            y_pred=y_pred_real,
            variable_name=col,
            model_name="MLP",
            folder_name="08_residMLP"
        )
    except Exception as e:
        print(f"Residual plot failed: {e}")
    
    rep.save_experiment_results(
        indicator=ind,
        model_name="MLP",
        configuration=f"{sampler}_{ds_type}",
        y_test=y_test_real,
        y_pred=y_pred_real,
        years_test=aligned_years,
        params=params,
        training_time=elapsed_time
    )
    
    print("DONE")

Refitting on 12 configurations.

Refitting: cerealland_abs | Random | orig ...
  -> Residuals: Mean=-1045831.7500, Shapiro P=0.5881, Ljung-Box P=0.2130
Residual plot failed: The data contains non-finite values.
Saving results for MLP | cerealland_abs...
Leaderboard updated: cerealland_abs | MLP
Save leaderboard complete.
DONE

Refitting: cerealland_abs | Random | step_aug ...
  -> Residuals: Mean=-86594.0833, Shapiro P=0.0968, Ljung-Box P=0.7048
Residual plot failed: The data contains non-finite values.
Saving results for MLP | cerealland_abs...
Leaderboard updated: cerealland_abs | MLP
Save leaderboard complete.
DONE

Refitting: cerealland_abs | Random | jitter_aug ...
  -> Residuals: Mean=-796806.8000, Shapiro P=0.2034, Ljung-Box P=0.2196
Residual plot failed: The data contains non-finite values.
Saving results for MLP | cerealland_abs...
Leaderboard updated: cerealland_abs | MLP
Save leaderboard complete.
DONE

Refitting: food_production_index | Random | orig ...
  -> Residuals: Mea

c:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\.venv\Lib\site-packages\matplotlib\axes\_axes.py:7104: RuntimeWarning: All-NaN slice encountered
  xmin = min(xmin, np.nanmin(xi))
c:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\.venv\Lib\site-packages\matplotlib\axes\_axes.py:7105: RuntimeWarning: All-NaN slice encountered
  xmax = max(xmax, np.nanmax(xi))


Leaderboard updated: food_production_index | MLP
Save leaderboard complete.
DONE

Refitting: food_production_index | Random | step_aug ...
  -> Residuals: Mean=nan, Shapiro P=nan, Ljung-Box P=nan
Residual plot failed: autodetected range of [nan, nan] is not finite
Saving results for MLP | food_production_index...


c:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\.venv\Lib\site-packages\matplotlib\axes\_axes.py:7104: RuntimeWarning: All-NaN slice encountered
  xmin = min(xmin, np.nanmin(xi))
c:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\.venv\Lib\site-packages\matplotlib\axes\_axes.py:7105: RuntimeWarning: All-NaN slice encountered
  xmax = max(xmax, np.nanmax(xi))


Leaderboard updated: food_production_index | MLP
Save leaderboard complete.
DONE

Refitting: food_production_index | Random | jitter_aug ...
  -> Residuals: Mean=nan, Shapiro P=nan, Ljung-Box P=nan
Residual plot failed: autodetected range of [nan, nan] is not finite
Saving results for MLP | food_production_index...


...

  -> Residuals: Mean=nan, Shapiro P=nan, Ljung-Box P=nan
Residual plot failed: autodetected range of [nan, nan] is not finite
Saving results for MLP | food_production_index...


c:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\.venv\Lib\site-packages\matplotlib\axes\_axes.py:7104: RuntimeWarning: All-NaN slice encountered
  xmin = min(xmin, np.nanmin(xi))
c:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\.venv\Lib\site-packages\matplotlib\axes\_axes.py:7105: RuntimeWarning: All-NaN slice encountered
  xmax = max(xmax, np.nanmax(xi))


Leaderboard updated: food_production_index | MLP
Save leaderboard complete.
DONE

Refitting: food_production_index | TPE | step_aug ...
  -> Residuals: Mean=0.1781, Shapiro P=0.7662, Ljung-Box P=0.9783
Residual plot failed: The data contains non-finite values.
Saving results for MLP | food_production_index...
Leaderboard updated: food_production_index | MLP
Save leaderboard complete.
DONE

Refitting: food_production_index | TPE | jitter_aug ...
  -> Residuals: Mean=nan, Shapiro P=nan, Ljung-Box P=nan
Residual plot failed: autodetected range of [nan, nan] is not finite
Saving results for MLP | food_production_index...


c:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\.venv\Lib\site-packages\matplotlib\axes\_axes.py:7104: RuntimeWarning: All-NaN slice encountered
  xmin = min(xmin, np.nanmin(xi))
c:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\.venv\Lib\site-packages\matplotlib\axes\_axes.py:7105: RuntimeWarning: All-NaN slice encountered
  xmax = max(xmax, np.nanmax(xi))


Leaderboard updated: food_production_index | MLP
Save leaderboard complete.
DONE


- ACF : barre devono essere basse e dentro l'area azzurra
- residui nel tempo : i punti devono oscillare a caso attorno allo zero, senza pattern
- istogramma : deve sembrare una campana attorno allo zero

L'analisi dei residui mostra una significativa autocorrelazione al Lag 1. Questo suggerisce che il modello MLP, nonostante l'ottimizzazione, non è riuscito a catturare completamente la dinamica a breve termine. Possibili miglioramenti futuri includono l'aumento della finestra temporale (n_input_steps) o l'utilizzo di reti ricorrenti (LSTM) che gestiscono meglio la memoria sequenziale.

L'analisi dei residui mostra che gli errori del modello MLP si approssimano al Rumore Bianco (assenza di autocorrelazione significativa). Ciò indica che il modello ha catturato correttamente la componente deterministica della serie storica. L'errore residuo è attribuibile alla volatilità intrinseca e stocastica dei dati macroeconomici, non a una mancanza di capacità del modello.

L'analisi diagnostica dei residui conferma la validità del training. I plot di autocorrelazione (ACF) mostrano che i residui si approssimano al Rumore Bianco, indicando che il modello MLP ha catturato con successo tutta la componente deterministica e i pattern lineari/non-lineari presenti nella finestra temporale considerata. L'errore residuo è pertanto attribuibile alla componente stocastica (imprevedibile) intrinseca nei dati macroeconomici, e non a un difetto di apprendimento del modello.

In [32]:
for indicator, samplers in predictions_dict.items():
    for sampler, datasets in samplers.items():
        for ds_type, res in datasets.items():
            y_true = res.get('y_true')
            y_pred = res.get('y_pred')
            if y_true is None or y_pred is None:
                print(f"SKIP {indicator} | {sampler} | {ds_type}: missing y_true or y_pred")
                continue
            try:
                resid = eval.compute_residual_diagnostics(y_true, y_pred)
                print(f"{indicator} | {sampler} | {ds_type} - residuals mean: {resid['residual_mean']:.4f}, std: {resid['residual_std']:.4f}")
                visual.plot_residuals(
                    y_true=y_true,
                    y_pred=y_pred,
                    variable_name=indicator,
                    model_name=f"{sampler}_{ds_type}",
                    folder_name="08_residMLP"
                ) 
            except Exception as e:
                print(f"Error with {indicator} | {sampler} | {ds_type}: {e}")
                continue

cerealland_abs | Random | orig - residuals mean: -1045831.7500, std: 41516.5738
Error with cerealland_abs | Random | orig: The data contains non-finite values.
cerealland_abs | Random | step_aug - residuals mean: -86594.0833, std: 35707.7838
Error with cerealland_abs | Random | step_aug: The data contains non-finite values.
cerealland_abs | Random | jitter_aug - residuals mean: -796806.8000, std: 60024.4370
Error with cerealland_abs | Random | jitter_aug: The data contains non-finite values.
cerealland_abs | TPE | orig - residuals mean: -1052824.7500, std: 41516.5738
Error with cerealland_abs | TPE | orig: The data contains non-finite values.
cerealland_abs | TPE | step_aug - residuals mean: nan, std: nan
Error with cerealland_abs | TPE | step_aug: autodetected range of [nan, nan] is not finite
cerealland_abs | TPE | jitter_aug - residuals mean: nan, std: nan
Error with cerealland_abs | TPE | jitter_aug: autodetected range of [nan, nan] is not finite
food_production_index | Random | or

c:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\.venv\Lib\site-packages\matplotlib\axes\_axes.py:7104: RuntimeWarning: All-NaN slice encountered
  xmin = min(xmin, np.nanmin(xi))
c:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\.venv\Lib\site-packages\matplotlib\axes\_axes.py:7105: RuntimeWarning: All-NaN slice encountered
  xmax = max(xmax, np.nanmax(xi))


Error with food_production_index | Random | orig: autodetected range of [nan, nan] is not finite
food_production_index | Random | step_aug - residuals mean: nan, std: nan
Error with food_production_index | Random | step_aug: autodetected range of [nan, nan] is not finite
food_production_index | Random | jitter_aug - residuals mean: nan, std: nan
Error with food_production_index | Random | jitter_aug: autodetected range of [nan, nan] is not finite
food_production_index | TPE | orig - residuals mean: nan, std: nan
Error with food_production_index | TPE | orig: autodetected range of [nan, nan] is not finite
food_production_index | TPE | step_aug - residuals mean: 0.1781, std: 2.2084
Error with food_production_index | TPE | step_aug: The data contains non-finite values.
food_production_index | TPE | jitter_aug - residuals mean: nan, std: nan
Error with food_production_index | TPE | jitter_aug: autodetected range of [nan, nan] is not finite


In [19]:
for indicator in INDICATORS:
    if indicator not in orig_aug_subsets:
        print(f"SKIP {indicator}: missing orig_aug_subsets")
        continue

    try:
        train_orig = orig_aug_subsets[indicator]['orig_train']
        val_orig = orig_aug_subsets[indicator]['orig_val']
        test_orig = orig_aug_subsets[indicator]['orig_test']
    except KeyError:
        print(f"SKIP {indicator}: missing one of orig_train/orig_val/orig_test")
        continue

    if indicator not in predictions_dict:
        print(f"SKIP {indicator}: no predictions available")
        continue

    print(f"\nPlotting predictions for: {indicator}")
    visual.plot_nn_preds(
        indicator,
        predictions_dict,
        train_orig,
        val_orig,
        test_orig,
        model_name="MLP",
        folder_name="08_MLP"
    )

    for sampler, cats in predictions_dict[indicator].items():
        for cat, res in cats.items():
            yrs = res.get("years")
            vals = res.get("y_pred")
            if yrs is None or vals is None:
                print(f"{sampler} | {cat}: missing years or preds")
            else:
                print(f"{sampler} | {cat} -> years: {yrs.tolist()} preds: {np.round(vals,4).tolist()}")


Plotting predictions for: cerealland_abs
Grafico salvato in: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\08_MLP\MLP_cerealland_abs.png
Random | orig -> years: [2018, 2019, 2020, 2021, 2022, 2023, 2024] preds: [4077823.5, 4077823.5, 4077823.5, 4077823.5, 4077823.5, 4077823.5, 4077823.5]
Random | step_aug -> years: [2020, 2021, 2022, 2023, 2024] preds: [3125756.5, 3087985.25, 3046780.5, 3047340.75, 4747739.0]
Random | jitter_aug -> years: [2018, 2019, 2020, 2021, 2022, 2023, 2024] preds: [3810456.0, 3798554.5, 3851783.75, 3843403.25, 3839796.0, 3898541.75, 4218916.0]
TPE | orig -> years: [2018, 2019, 2020, 2021, 2022, 2023, 2024] preds: [4084816.75, 4084816.75, 4084816.75, 4084816.75, 4084816.75, 4084816.75, 4084816.75]
TPE | step_aug -> years: [2023, 2024] preds: [2839772.0, 4523955.0]
TPE | jitter_aug -> years: [2023, 2024] preds: [4059218.25, 4220303.0]

Plotting predictions for: food_production_index
Grafico salvato in: C:\Users\oldan\Desktop\RuralDevelo

## <a id='toc1_8_'></a>[Future preds - SANITIZED TAIL METHOD](#toc0_)

qua invece ho preso tutta la time serie originale e l'ho aumentata tutta tranne gli utlimi 5 anni di validation set che ho lasciato puri

In [20]:
future_predictions_dict2 = {}
N_VAL_YEARS = 5

for indicator in INDICATORS:
    LAST_DATA_YEAR = int(orig_aug_subsets[indicator]['full_orig']['Year'].iloc[-1])
    TARGET_YEAR = 2030
    future_years = np.arange(LAST_DATA_YEAR + 1, TARGET_YEAR + 1)
    STEPS_AHEAD = len(future_years)

    if indicator not in future_predictions_dict2:
        future_predictions_dict2[indicator] = {}

    print(f"Forecasting for period: {future_years[0]}-{future_years[-1]}")
    for entry in tuning_results:
        if entry['indicator'] != indicator:
            continue

        ind = entry['indicator']
        cat = entry['dataset_type']
        params = entry['best_params']
        sampler = entry['sampler']
        
        if sampler not in future_predictions_dict2[indicator]:
            future_predictions_dict2[indicator][sampler] = {}
        
        print(f"Forecasting {ind} ({sampler}, {cat})")
        full_orig = orig_aug_subsets[ind]['full_orig'].sort_values('Year')
        # full_values = full_orig['Value'].values
        
        split_idx = len(full_orig) - N_VAL_YEARS
        orig_train_set = full_orig.iloc[:split_idx]
        orig_val_set = full_orig.iloc[split_idx:]
        
        x_train_vals = orig_train_set['Year']
        y_train_vals = orig_train_set['Value'].values
        
        if cat == 'step_aug':
            step_train_set = pipe.augment_step_function(x_train_vals, y_train_vals, scale_factor=10) 
            final_train_set = step_train_set['Value'].values
        elif cat == 'jitter_aug':
            jitter_train_set = pipe.augment_linear_with_jitter(x_train_vals, y_train_vals, scale_factor=10, noise_level=0.05)
            final_train_set = jitter_train_set['Value'].values
        else: # 'orig' -> Nessuna augmentation
            final_train_set = orig_train_set['Value'].values
        
        print(f"-> Training on {len(final_train_set)} years (original/augmented) + {len(orig_val_set)} years (pure for validation set)")
        
        full_values_raw = np.concatenate([final_train_set, orig_val_set['Value'].values])
        
        scaler = MinMaxScaler(feature_range=(0, 1))
        full_values_scaled = scaler.fit_transform(full_values_raw.reshape(-1, 1)).flatten()
        
        n_steps = params['n_input_steps']
        X_full, y_full = pipe.create_sequences(full_values_scaled , n_steps)
        
        model = nn.build_mlp_model(params, input_dim=n_steps)

        val_slice_len = len(orig_val_set) + n_steps
        val_set_scaled_slice = full_values_scaled[-val_slice_len:]
        X_val, y_val = pipe.create_sequences(val_set_scaled_slice, n_steps)
    
        # se la coda è troppo corta per creare sequenze, usiamo X_full (Self-Validation)
        if len(X_val) == 0:
            X_val_final, y_val_final = X_full, y_full
        else:
            X_val_final, y_val_final = X_val, y_val

        history, model = nn.train_model(
            model, 
            X_train=X_full, y_train=y_full, 
            X_val=X_val_final, y_val=y_val_final,
            batch_size=params['batch_size'],
            epochs=150, 
            patience=20
        )
        
        last_window_scaled = full_values_scaled[-n_steps:]
        future_preds_scaled = nn.recursive_forecast(model, last_window_scaled, STEPS_AHEAD)
        
        future_preds_real = scaler.inverse_transform(future_preds_scaled.reshape(-1, 1)).flatten()
        future_predictions_dict2[indicator][sampler][cat] = {
            "years": future_years,
            "y_pred": future_preds_real
        }

Forecasting for period: 2023-2030
Forecasting cerealland_abs (Random, orig)
-> Training on 57 years (original/augmented) + 5 years (pure for validation set)
Forecasting cerealland_abs (Random, step_aug)
-> Training on 570 years (original/augmented) + 5 years (pure for validation set)
Forecasting cerealland_abs (Random, jitter_aug)
-> Training on 570 years (original/augmented) + 5 years (pure for validation set)
Forecasting cerealland_abs (TPE, orig)
-> Training on 57 years (original/augmented) + 5 years (pure for validation set)
Forecasting cerealland_abs (TPE, step_aug)
-> Training on 570 years (original/augmented) + 5 years (pure for validation set)
Forecasting cerealland_abs (TPE, jitter_aug)
-> Training on 570 years (original/augmented) + 5 years (pure for validation set)
Forecasting for period: 2023-2030
Forecasting food_production_index (Random, orig)
-> Training on 57 years (original/augmented) + 5 years (pure for validation set)
Forecasting food_production_index (Random, step_a

In [28]:
for indicator in future_predictions_dict2:
    try:
        train_orig = orig_aug_subsets[indicator]['orig_train']
        val_orig = orig_aug_subsets[indicator]['orig_val']
        test_orig = orig_aug_subsets[indicator]['orig_test']
    except KeyError:
        print(f"Skipping {indicator}: missing subsets")
        continue

    print(f"\nPlotting future for: {indicator}")
    
    visual.plot_nn_preds(
        variable_name=indicator,
        predictions_dict=future_predictions_dict2,
        train_df=train_orig,
        val_df=val_orig,
        test_df=test_orig,
        model_name="MLP",
        folder_name="08_futureMLP"
    )

    for sampler, cats in future_predictions_dict2[indicator].items():
        for cat, pred in cats.items():
            yrs = pred["years"]
            vals = pred["y_pred"]
            print(f"{indicator} | {sampler} | {cat} -> years: {yrs.tolist()} preds: {np.round(vals, 4).tolist()}")


Plotting future for: cerealland_abs
Grafico salvato in: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\08_futureMLP\MLP_cerealland_abs.png
cerealland_abs | Random | orig -> years: [2023, 2024, 2025, 2026, 2027, 2028, 2029, 2030] preds: [3016511.25, 3017607.75, 3021959.25, 3022452.5, 3022824.0, 3023059.0, 3023114.0, 3023147.0]
cerealland_abs | Random | step_aug -> years: [2023, 2024, 2025, 2026, 2027, 2028, 2029, 2030] preds: [3165288.5, 3172183.75, 3189862.0, 3219944.0, 3256118.25, 3279747.75, 3297490.5, 3314847.5]
cerealland_abs | Random | jitter_aug -> years: [2023, 2024, 2025, 2026, 2027, 2028, 2029, 2030] preds: [2961874.75, 2963344.75, 2931590.25, 2928673.75, 2907191.75, 2904124.75, 2891824.25, 2892451.25]
cerealland_abs | TPE | orig -> years: [2023, 2024, 2025, 2026, 2027, 2028, 2029, 2030] preds: [3070700.0, 3057049.75, 3066580.75, 3070905.75, 3067607.75, 3067942.0, 3068777.5, 3068359.5]
cerealland_abs | TPE | step_aug -> years: [2023, 2024, 2025, 2026

-------------

## <a id='toc1_7_'></a>[Future preds - REFITTING ON FULL HISTORY WITH SANITIZED TAIL](#toc0_)

qua avevo concatenato il train originale o aumentato con validation e test set base, ma mi sa che è stupido

In [ ]:
future_predictions_dict1 = {}

for indicator in INDICATORS:
    LAST_DATA_YEAR = int(orig_aug_subsets[indicator]['full_orig']['Year'].iloc[-1])
    TARGET_YEAR = 2030
    future_years = np.arange(LAST_DATA_YEAR + 1, TARGET_YEAR + 1)
    STEPS_AHEAD = len(future_years)

    if indicator not in future_predictions_dict1:
        future_predictions_dict1[indicator] = {}

    print(f"Forecasting for period: {future_years[0]}-{future_years[-1]}")
    for entry in tuning_results:
        if entry['indicator'] != indicator:
            continue

        ind = entry['indicator']
        cat = entry['dataset_type']
        params = entry['best_params']
        sampler = entry['sampler']
        
        if sampler not in future_predictions_dict1[indicator]:
            future_predictions_dict1[indicator][sampler] = {}
        
        print(f"Forecasting {ind} ({sampler}, {cat})")
        train_part = scaled_orig_aug_subsets[ind][f'{cat}_train']
        val_part = scaled_orig_aug_subsets[ind]['orig_val']
        test_part = scaled_orig_aug_subsets[ind]['orig_test']
        
        full_values_scaled = np.concatenate([
            train_part['Value'].values,
            val_part['Value'].values,
            test_part['Value'].values
        ])
        
        n_steps = params['n_input_steps']
        X_full, y_full = pipe.create_sequences(full_values_scaled, n_steps)
        
        model = nn.build_mlp_model(params, input_dim=n_steps)
        
        history, model = nn.train_model(
            model, 
            X_train=X_full, 
            y_train=y_full, 
            X_val=X_full,           # self validation : chiedere se ok
            y_val=y_full,
            batch_size=params['batch_size'],
            epochs=150,
            patience=20
        )    
        
        last_window_scaled = full_values_scaled[-n_steps:]
        future_preds_scaled = nn.recursive_forecast(model, last_window_scaled, STEPS_AHEAD)
        
        scaler_y = scalers_subsets[ind]['scaler_y']
        future_preds_real = scaler_y.inverse_transform(future_preds_scaled.reshape(-1, 1)).flatten()
        
        future_predictions_dict1[indicator][sampler][cat] = {
            "years": future_years,
            "y_pred": future_preds_real
        }

Forecasting for period: 2025-2030
Forecasting agriland_percent (Random, orig)
Forecasting agriland_percent (TPE, orig)
Forecasting for period: 2025-2030
Forecasting arableland_percent (Random, orig)
Forecasting arableland_percent (TPE, orig)


In [ ]:
for indicator in future_predictions_dict1:
    try:
        train_orig = orig_aug_subsets[indicator]['orig_train']
        val_orig = orig_aug_subsets[indicator]['orig_val']
        test_orig = orig_aug_subsets[indicator]['orig_test']
    except KeyError:
        print(f"Skipping {indicator}: missing original subsets")
        continue

    print(f"\nPlotting future forecast for: {indicator}")
    # visual.plot_future_forecasts(
    #     full_history=,
    #     future_pred=,
    #     baseline_pred=,
    #     variable_name=indicator,
    #     baseline_name=,
    #     folder_name="08_futureMLP"
    # )

    # print corresponding future years and predictions for clarity
    for sampler, cats in future_predictions_dict1[indicator].items():
        for cat, pred in cats.items():
            yrs = pred["years"]
            vals = pred["y_pred"]
            print(f"{indicator} | {sampler} | {cat} -> years: {yrs.tolist()} preds: {np.round(vals, 4).tolist()}")


Plotting future forecast for: agriland_percent


TypeError: plot_future_forecasts() got multiple values for argument 'model_name'